# Distributed Multi-agent solution in Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to independently host agents each in their own Bedrock AgentCore Runtime and built with different Agentic Frameworks. We'll then enable communication between them for a distributed multi-agent solution. 

In this example we'll create:
1. A technical agent (`tech_agent`) that is specialized in answering technical questions about programming and tech troubleshooting.
2. A HR agent (`hr_agent`) that is specialized in company benefits.
3. An orchestrator agent (`orchestrator_agent`) that routes questions to the technical or HR agent.

Putting these three agents together you get a multi-agent configuration with a supervisor, which can route user questions to the appropriate subagent. This system is capable of answering a range of questions an employee might have at a company.


### Tutorial Details


| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                                   |
| Agent type          | Multi-Agent (Supervisor calling agents as tools)                                                                           |
| Agentic Framework   | Strands Agents & LangGraph                                                                  |
| LLM model           | Anthropic Claude Haiku 4.5                                                        |
| Tutorial components | Hosting agents on AgentCore Runtime and enable multi-agent collaboration |
| Tutorial vertical   | Cross-vertical                                                                   |
| Example complexity  | Medium                                                                             |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                                     |

### Tutorial Architecture

In this tutorial we will describe how to deploy 3 agents to Bedrock AgentCore runtime. We will use a Strands Agent for the Orchestrator, a Strands Agent for the Tech agent, and a LangGraph agent for the HR agent. We will use simple agents to demonstrate how you can configure a multi-agent system with a mix of agent frameworks, with each agent deployed to it's own AgentCore Runtime.

![alt text](./architecture.png)


### Tutorial Key Features

* Hosting multiple Agents on Amazon Bedrock AgentCore Runtime
* Creating a Multi-agent solution where each agent is hosting independently


## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents
* LangGraph

In [13]:
!uv pip install --force-reinstall -U -r requirements.txt --quiet

error: File not found: `requirements.txt`


In [1]:
import os

# Set an environment variable
os.environ["AWS_DEFAULT_REGION"] = "us-west-2"
print(os.getcwd())


/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3


In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
user_name = os.getenv("USER_NAME")
if not user_name:
    raise ValueError("USER_NAME environment variable is not set. Please set it in the .env file.")


## Creating our Agents

First we will create three separate IAM roles for each agent. This enables us to define least privilege permissions for each agent independently of the others.

In [3]:
from utils import create_agentcore_role

tech_agent_name="tech_agent_" + user_name
tech_agent_iam_role = create_agentcore_role(agent_name=tech_agent_name, region=os.getenv("AWS_DEFAULT_REGION"))
tech_agent_role_arn = tech_agent_iam_role['Role']['Arn']
tech_agent_role_name = tech_agent_iam_role['Role']['RoleName']
print(tech_agent_role_arn)
print(tech_agent_role_name)

hr_agent_name="hr_agent_" + user_name
hr_agent_iam_role = create_agentcore_role(agent_name=hr_agent_name, region=os.getenv("AWS_DEFAULT_REGION"))
hr_agent_role_arn = hr_agent_iam_role['Role']['Arn']
hr_agent_role_name = hr_agent_iam_role['Role']['RoleName']
print(hr_agent_role_arn)
print(hr_agent_role_name)

orchestrator_agent_name="orchestrator_agent_" + user_name
orchestrator_iam_role = create_agentcore_role(agent_name=orchestrator_agent_name, region=os.getenv("AWS_DEFAULT_REGION"))
orchestrator_role_arn = orchestrator_iam_role['Role']['Arn']
orchestrator_role_name = orchestrator_iam_role['Role']['RoleName']
print(orchestrator_role_arn)
print(orchestrator_role_name)

Role already exists -- deleting and creating it again
policies: {'PolicyNames': ['AgentCorePolicy'], 'IsTruncated': False, 'ResponseMetadata': {'RequestId': '572e535b-ba3f-4898-a20f-a7736c6f7b09', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sun, 05 Apr 2026 23:51:24 GMT', 'x-amzn-requestid': '572e535b-ba3f-4898-a20f-a7736c6f7b09', 'content-type': 'text/xml', 'content-length': '380'}, 'RetryAttempts': 0}}
deleting agentcore-tech_agent_eric_fu-role
recreating agentcore-tech_agent_eric_fu-role
attaching role policy agentcore-tech_agent_eric_fu-role
arn:aws:iam::372080370602:role/agentcore-tech_agent_eric_fu-role
agentcore-tech_agent_eric_fu-role
Role already exists -- deleting and creating it again
policies: {'PolicyNames': ['AgentCorePolicy'], 'IsTruncated': False, 'ResponseMetadata': {'RequestId': '8c84d04b-a6d7-4c3b-b1ef-1dc96e4794a3', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Sun, 05 Apr 2026 23:51:25 GMT', 'x-amzn-requestid': '8c84d04b-a6d7-4c3b-b1ef-1dc96e4794a3', 'content

### Helper Functions:

* the `configure_runtime` helper function will be used to setup the runtime configurate for each agent. In this example, we use the starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.
* the `check_status` helper function will be used to check each runtime deployed in the AWS account to validate the creation was successful and the agent is ready to be used.

In [4]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import time


def configure_runtime(agent_name, agentcore_iam_role, python_file_name):
    boto_session = Session(region_name=os.getenv("AWS_DEFAULT_REGION"))
    region = boto_session.region_name

    agentcore_runtime = Runtime()

    response = agentcore_runtime.configure(
        entrypoint=python_file_name,
        execution_role=agentcore_iam_role['Role']['Arn'],
        auto_create_ecr=True,
        requirements_file="requirements.txt",
        region=region,
        agent_name=agent_name
    )
    return response, agentcore_runtime

def check_status(agent_runtime):
    status_response = agent_runtime.status()
    status = status_response.endpoint['status']
    end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
    while status not in end_status:
        time.sleep(10)
        status_response = agent_runtime.status()
        status = status_response.endpoint['status']
        print(status)
    return status

In [5]:
# set the current working directory to be the tech_agent folder
import os
os.chdir('./tech_agent')
print(os.getcwd())

/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/tech_agent


### Create Tech Support Agent (Strands Agents)

Let's start with the Tech Support Agent using Strands and an Amazon Bedrock model. Executing the following cell will create the `tech_agent.py` file in the `./tech_agent` directory with the agent specific logic. 

Note the app is defined with `BedrockAgentCoreApp()` and the invocation function `strands_agent_bedrock` is decorated with the `@app.entrypoint` decorator, and the `app.run()` command is at the end of the file.

In [6]:
%%writefile tech_agent.py

from strands import Agent, tool
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel

app = BedrockAgentCoreApp()

model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    system_prompt="You're a helpful tech support assistant, you can help user questions on tech troubleshooting and programming"
)

@app.entrypoint
def strands_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

Overwriting tech_agent.py


#### Launch the agent:

First, we use the configure_runtime helper function to create the .bedrock_agentcore.yaml, .dockerignore, and Dockerfile required for the agent deployment. Then we call .launch() on the runtime which pushes the image to ECR and creates the AgentCore Runtime in the AWS environment. 


In [7]:
_, tech_agent_runtime = configure_runtime("tech_agent_" + user_name, tech_agent_iam_role, "tech_agent.py")
tech_launch_result = tech_agent_runtime.launch()
tech_agent_id = tech_launch_result.agent_id
tech_agent_arn = tech_launch_result.agent_arn

print(tech_agent_arn)

Entrypoint parsed: file=/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/tech_agent/tech_agent.py, bedrock_agentcore_name=tech_agent
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: tech_agent_eric_fu
Memory disabled
Network mode: PUBLIC


📄 Using existing Dockerfile: 
/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-ag
ents/01-multi-runtimes-with-boto3/tech_agent/Dockerfile

Generated .dockerignore: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/tech_agent/.dockerignore
Keeping 'tech_agent_eric_fu' as default agent
Bedrock AgentCore configured: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/tech_agent/.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'tech_agent_eric_fu' to account 372080370602 (us-west-2)
Generated image tag: 20260405-235144-806
Setting up AWS resources (ECR repositor

✅ Reusing existing ECR repository: 372080370602.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-tech_agent_eric_fu


Reusing existing CodeBuild execution role: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-f790ae425e
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: tech_agent_eric_fu/source.zip
Created CodeBuild project: bedrock-agentcore-tech_agent_eric_fu-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.4s
🔄 DOWNLOAD_SOURCE started (total: 9s)
✅ DOWNLOAD_SOURCE completed in 1.1s
🔄 BUILD started (total: 10s)
✅ BUILD completed in 18.1s
🔄 POST_BUILD started (total: 28s)
✅ POST_BUILD completed in 12.7s
🔄 COMPLETED started (total: 40s)
✅ COMPLETED completed in 1.1s
🎉 CodeBuild completed successfully in 0m 41s
CodeBuild completed successfully
CodeBuild project configuration saved
Deploying to Bedrock AgentCore...
⚠️ Session ID will be reset to connect to the updated ag

arn:aws:bedrock-agentcore:us-west-2:372080370602:runtime/tech_agent_eric_fu-SQgUotG3ed


#### Let's save the Tech Agent ARN to parameter store 

This creates a simple agent registry so we can persistently store and look up the AgentCore Runtime ARN for the Tech Agent

In [8]:
import boto3
import time

ssm = boto3.client('ssm')
ssm.put_parameter(
    Name=f'/agents/tech_agent_arn_{user_name}',
    Value=tech_agent_arn,
    Type='String',
    Overwrite=True
)

{'Version': 2,
 'Tier': 'Standard',
 'ResponseMetadata': {'RequestId': '39b9b489-f047-4bc0-9ce7-779e371a803e',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Server',
   'date': 'Sun, 05 Apr 2026 23:52:45 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '31',
   'connection': 'keep-alive',
   'x-amzn-requestid': '39b9b489-f047-4bc0-9ce7-779e371a803e',
   'cache-control': 'no-store'},
  'RetryAttempts': 0}}

#### Test the agent

To test the agent, let's first check the status of the Tech Agent AgentCore Runtime and confirm it is ready for use.\
Use `.invoke()` on the tech_agent_runtime to validate the agent runtime is configured and working as expected.

In [9]:
status = check_status(tech_agent_runtime)
print(status)

Retrieved Bedrock AgentCore status for: tech_agent_eric_fu


READY


In [10]:
invoke_response = tech_agent_runtime.invoke({"prompt": "shortcut to minimize windows in Mac, in 1 sentence"})
invoke_response

{'ResponseMetadata': {'RequestId': '49b9d952-d72a-47e1-954b-04039890870a',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 05 Apr 2026 23:53:03 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': '49b9d952-d72a-47e1-954b-04039890870a',
   'x-amzn-bedrock-agentcore-runtime-session-id': '81920de2-6784-4633-aaa4-b7615ec496fc'},
  'RetryAttempts': 0},
 'runtimeSessionId': '81920de2-6784-4633-aaa4-b7615ec496fc',
 'contentType': 'application/json',
 'statusCode': 200,
 'response': ['Press **Command + M** to minimize the active window in macOS.']}

### Create HR Agent (Langraph Agents)

We will follow a similar process to create the HR Agent. However, this time you will notice the underlying agent logic is build with LangGraph. This change in Agent Framework does not have an impact on how we configure the AgentCore Runtime. Executing the following cell will create the `hr_agent.py`file in the `./hr_agent` directory.

Just as with the Tech Support Agent, we define the app the `BedrockAgentCoreApp()` and the invocation function langgraph_bedrock is decorated with the @app.entrypoint decorator, and the `app.run()` command is at the end of the file. 

In [11]:
# set the current working directory to be the hr_agent folder
import os

os.chdir('../hr_agent')
print(os.getcwd())

/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/hr_agent


In [12]:
%%writefile hr_agent.py
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from bedrock_agentcore.runtime import BedrockAgentCoreApp
import argparse
import json
import operator
import math

app = BedrockAgentCoreApp()

@tool
def get_vacation_info():
    """Get remaining vacation days balance for the current year"""  # Dummy implementation
    return "you have 12 days off remaining this year"

# Define the agent using manual LangGraph construction
def create_agent():
    """Create and configure the LangGraph agent"""
    from langchain_aws import ChatBedrock

    # Initialize your LLM (adjust model and parameters as needed)
    llm = ChatBedrock(
        model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # or your preferred model
        model_kwargs={"temperature": 0.1}
    )

    # Bind tools to the LLM
    tools = [get_vacation_info]
    llm_with_tools = llm.bind_tools(tools)

    # System message
    system_message = f"""You're a helpful hr support assistant, you can answers user questions on vacations and benefits.
    Here are the primary company benefits
    - Comprehensive health insurance with 100% premium coverage for employees and 75% for dependents
    - Flexible PTO policy with 20 days paid vacation annually, plus 5 sick days
    - 401(k) plan with 6% company matching and immediate vesting
    - Monthly wellness stipend of $100 for gym memberships or fitness activities

    For additional HR information instruct the user to call to 1-800-ASKHR"""

    # Define the chatbot node
    def chatbot(state: MessagesState):
        # Add system message if not already present
        messages = state["messages"]
        if not messages or not isinstance(messages[0], SystemMessage):
            messages = [SystemMessage(content=system_message)] + messages

        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}

    # Create the graph
    graph_builder = StateGraph(MessagesState)

    # Add nodes
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))

    # Add edges
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")

    # Set entry point
    graph_builder.set_entry_point("chatbot")

    # Compile the graph
    return graph_builder.compile()

# Initialize the agent
agent = create_agent()

@app.entrypoint
def langgraph_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")

    # Create the input in the format expected by LangGraph
    response = agent.invoke({"messages": [HumanMessage(content=user_input)]})

    # Extract the final message content
    return response["messages"][-1].content

if __name__ == "__main__":
    app.run()

Overwriting hr_agent.py


#### Launch the agent:

Again, we use the configure_runtime helper function to create the .bedrock_agentcore.yaml, .dockerignore, and Dockerfile required for the agent deployment. Then we call .launch() on the hr agent runtime which pushes the image to ECR and creates the AgentCore Runtime in the AWS environment. 

In [15]:
_, hr_agentcore_runtime = configure_runtime("hr_agent_" + user_name, hr_agent_iam_role, "hr_agent.py")
hr_launch_result = hr_agentcore_runtime.launch()
hr_agent_id = hr_launch_result.agent_id
hr_agent_arn = hr_launch_result.agent_arn

print(hr_agent_arn)

Entrypoint parsed: file=/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/hr_agent/hr_agent.py, bedrock_agentcore_name=hr_agent
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: hr_agent_eric_fu
Memory disabled
Network mode: PUBLIC


📄 Generated Dockerfile: 
/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-ag
ents/01-multi-runtimes-with-boto3/hr_agent/Dockerfile

Generated .dockerignore: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/hr_agent/.dockerignore
Setting 'hr_agent_eric_fu' as default agent
Bedrock AgentCore configured: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/hr_agent/.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'hr_agent_eric_fu' to account 372080370602 (us-west-2)
Generated image tag: 20260405-235858-827
Setting up AWS resources (ECR repository, execu

✅ Reusing existing ECR repository: 372080370602.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-hr_agent_eric_fu


Getting or creating CodeBuild execution role for agent: hr_agent_eric_fu
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-e21035c0ee
Reusing existing CodeBuild execution role: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-e21035c0ee
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: hr_agent_eric_fu/source.zip
Updated CodeBuild project: bedrock-agentcore-hr_agent_eric_fu-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.4s
🔄 DOWNLOAD_SOURCE started (total: 9s)
✅ DOWNLOAD_SOURCE completed in 1.1s
🔄 INSTALL started (total: 10s)
✅ INSTALL completed in 1.1s
🔄 BUILD started (total: 11s)
✅ BUILD completed in 18.0s
🔄 POST_BUILD started (total: 29s)
✅ POST_BUILD completed in 13.8s
🔄 COMPLETED started (total: 43s)
✅ COMPLETED completed in 1.1s
🎉 CodeBuild

arn:aws:bedrock-agentcore:us-west-2:372080370602:runtime/hr_agent_eric_fu-Vwau8T3bik


#### Let's save the HR Agent ARN to parameter store

This continues the simple agent registry so we can persistently store and look up the AgentCore Runtime ARN for the HR Agent

In [16]:
import boto3
import time

ssm = boto3.client('ssm')
ssm.put_parameter(
    Name=f'/agents/hr_agent_arn_{user_name}',
    Value=hr_agent_arn,
    Type='String',
    Overwrite=True
)

{'Version': 2,
 'Tier': 'Standard',
 'ResponseMetadata': {'RequestId': '241078e4-29a9-46d7-8947-01f3d7a5107a',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Server',
   'date': 'Mon, 06 Apr 2026 00:00:05 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '31',
   'connection': 'keep-alive',
   'x-amzn-requestid': '241078e4-29a9-46d7-8947-01f3d7a5107a',
   'cache-control': 'no-store'},
  'RetryAttempts': 0}}

#### Test the agent

Let's check the status of the HR Agent AgentCore Runtime and confirm it is ready for use.\
Use `.invoke()` on the hr_agentcore_runtime to validate the AgentCore runtime is configured and working as expected.

In [17]:
status = check_status(hr_agentcore_runtime)
status

Retrieved Bedrock AgentCore status for: hr_agent_eric_fu


'READY'

In [18]:
# Test your agent
invoke_response = hr_agentcore_runtime.invoke({"prompt": "How many vacation days I have left?"})
invoke_response

{'ResponseMetadata': {'RequestId': '05fb426d-e8c8-41cc-8379-440981131373',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Mon, 06 Apr 2026 00:00:27 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': '05fb426d-e8c8-41cc-8379-440981131373',
   'x-amzn-bedrock-agentcore-runtime-session-id': '70b268b2-a1a5-4e96-8656-861d4762dae5'},
  'RetryAttempts': 0},
 'runtimeSessionId': '70b268b2-a1a5-4e96-8656-861d4762dae5',
 'contentType': 'application/json',
 'statusCode': 200,
 'response': ["You have **12 vacation days remaining** for this year! \n\nAs a reminder, our company's flexible PTO policy provides:\n- **20 days** of paid vacation annually\n- **5 sick days** (separate from vacation days)\n\nIf you have any questions about scheduling your time off or need additional information about our benefits, feel free to ask or call 1-800-ASKHR for more detailed assistance."]}

### Create Orchestrator Agent (Strands Agents)

For our third agent, the orchestrator, let's use Strands for our Agent framework again. Before we create the agent, we need to update the AgentCore Runtime's execution role we created earlier to allow permissions for it to invoke the Tech Support Agent and the HR Agent.

The `update_orchestrator_permissions` function below takes in Arns of the sub agents and the Arns of the agents registered in Parameter Store and gives the orchestrator agent permission to invoke the DEFAULT runtime endpoint of the sub agents. It also gives the Orchestrator Agent permission to pull the Agent Arns from Parameter Store.

In [19]:
# Let's update the orchestrator agentcore exeuction role so it has permissions to invoke the required subagents
# the orchestrator also needs needs permissions to retrieve the sub agent arns from parameter store
import json

# retrieve the runtime arn from parameter store
ssm = boto3.client('ssm')
response = ssm.get_parameter(Name='/agents/tech_agent_arn_' + user_name)
tech_agent_arn = response['Parameter']['Value']
tech_agent_parameter_arn = response['Parameter']['ARN']

ssm = boto3.client('ssm')
response = ssm.get_parameter(Name='/agents/hr_agent_arn_' + user_name)
hr_agent_arn = response['Parameter']['Value']
hr_agent_parameter_arn = response['Parameter']['ARN']

def update_orchestrator_permissions(sub_agent_arns: list, sub_agent_parameter_arns: list, orchestrator_name: str):
    iam_client = boto3.client('iam')
    orchestrator_permissions = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "bedrock-agentcore:InvokeAgentRuntime"
                ],
                "Resource": [ sub_agent_arn + "/runtime-endpoint/DEFAULT" for sub_agent_arn in sub_agent_arns ] + [ sub_agent_arn for sub_agent_arn in sub_agent_arns ]
            },
            {
                "Effect": "Allow",
                "Action": [
                    "ssm:GetParameter"
                ],
                "Resource": [sub_agent_parameter_arn for sub_agent_parameter_arn in sub_agent_parameter_arns]

            }]
    }

    rsp = iam_client.put_role_policy(
        RoleName=orchestrator_name,
        PolicyName="subagent_permissions-new",
        PolicyDocument=json.dumps(orchestrator_permissions)
    )
    return rsp

rsp = update_orchestrator_permissions([tech_agent_arn, hr_agent_arn], [tech_agent_parameter_arn, hr_agent_parameter_arn], orchestrator_role_name)
print(rsp)

{'ResponseMetadata': {'RequestId': '25817c6d-f6e4-43a2-96f1-e1b663425a2c', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 06 Apr 2026 00:00:33 GMT', 'x-amzn-requestid': '25817c6d-f6e4-43a2-96f1-e1b663425a2c', 'content-type': 'text/xml', 'content-length': '206'}, 'RetryAttempts': 0}}


In [20]:
# set the current working directory to be the orchestrator_agent folder
import os
os.chdir('../orchestrator_agent')
print(os.getcwd())

/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/orchestrator_agent


Executing the following cell will create the `orchestrator_agent.py` file in the `./orchestrator_agent` directory with the agent specific logic. The orchestrator agent has two tools available to it: 1. `call_tech_agent` and 2. `call_HR_agent`. Both of these tools use the invoke_agent_utils function predefined in the `invoke_agent_utils.py` file already present in the `./orchestrator_agent` folder. The subagents are invoked as tools using the invoke_agent_runtime action available through boto3.

Even in this more complex agent setup, the Bedrock AgentCore Runtime configuration remains the same. Note the app is defined with `BedrockAgentCoreApp()` and the invocation function `strands_agent_bedrock_streaming` is decorated with the `@app.entrypoint` decorator, and the `app.run()` command is at the end of the file.

In [21]:
%%writefile orchestrator_agent.py

import argparse
import json
import boto3
import logging
from functools import partial

from strands import Agent, tool
from strands_tools import calculator
from strands.models import BedrockModel

from bedrock_agentcore.runtime import BedrockAgentCoreApp

from invoke_agent_utils import invoke_agent_with_boto3

logger = logging.getLogger(__name__)

app = BedrockAgentCoreApp()

def get_agent_arn(agent_name: str, user_name: str) -> str:
    """
    Retrieve agent ARN from Parameter Store
    """
    try:
        ssm = boto3.client('ssm')
        response = ssm.get_parameter(
            Name=f'/agents/{agent_name}_arn_{user_name}'
        )
        return response['Parameter']['Value']
    except Exception as err:
        print(err)
        raise err

def call_tech_agent(user_query, user_name):
    """ call the tech agent """
    # print("Calling tech agent")
    try:
        tech_agent_arn = get_agent_arn("tech_agent", user_name)
        result = invoke_agent_with_boto3(tech_agent_arn, user_query=user_query)
    except Exception as e:
        result = str(e)
        logger.exception("Exception calling tech agent: ")
    return result

def call_HR_agent(user_query, user_name):
    """ Get the HR agent """
    print("Calling HR agent")
    try:
        hr_agent_arn = get_agent_arn("hr_agent", user_name)
        print(hr_agent_arn)
        result = invoke_agent_with_boto3(hr_agent_arn, user_query=user_query)
    except Exception as e:
        result = str(e)
        logger.error(f"Exception calling hr agent: {e}", exc_info=True)
    return result


model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,

)

# Returns the agent with the tools. Note that the tools are wrapped with the tool decorator and partial to pass the user_name argument when invoked by the agent
def get_agent(user_name: str):
    call_tech_agent_partial = partial(call_tech_agent, user_name=user_name)
    call_tech_agent_partial.__name__ = "call_tech_agent"
    call_tech_agent_partial.__doc__ = "call the tech agent"
    call_HR_agent_partial = partial(call_HR_agent, user_name=user_name)
    call_HR_agent_partial.__name__ = "call_HR_agent"
    call_HR_agent_partial.__doc__ = "Get the HR agent"

    agent = Agent(
        model=model,
        system_prompt="You're a helpful assistant, your role is to understand user questions and delegate to the appropriate specialized agent, you have tools to call the tech and HR agents",
        tools=[tool(call_tech_agent_partial), tool(call_HR_agent_partial)]
    )
    return agent

def parse_event(event):
    """
    Parse a streaming event from the agent and return formatted output
    """
    # Skip events that don't need to be displayed
    if any(key in event for key in ['init_event_loop', 'start', 'start_event_loop']):
        return ""

    # Text chunks from supervisor
    if 'data' in event and isinstance(event['data'], str):
        return event['data']


    # Handle text messages from the assistant
    if 'event' in event:
        event_data = event['event']

        # Beginning of a tool use
        if 'contentBlockStart' in event_data and 'start' in event_data['contentBlockStart']:
            if 'toolUse' in event_data['contentBlockStart']['start']:
                tool_info = event_data['contentBlockStart']['start']['toolUse']
                return f"\n\n[Executing: {tool_info['name']}]\n\n"

    return ""

@app.entrypoint
async def strands_agent_bedrock_streaming(payload):
    """
    Invoke the agent with streaming capabilities
    This function demonstrates how to implement streaming responses
    with AgentCore Runtime using async generators
    """
    user_input = payload.get("prompt")
    user_name = payload.get("user_name")
    #print("User input:", user_input)
    agent = get_agent(user_name)
    try:
        # Stream each chunk as it becomes available
        async for event in agent.stream_async(user_input):
            text = parse_event(event)
            if text:  # Only return non-empty responses
                yield text

            #if "data" in event:
            #    yield event["data"]

    except Exception as e:
        # Handle errors gracefully in streaming context
        error_response = {"error": str(e), "type": "stream_error"}
        print(f"Streaming error: {error_response}")
        yield error_response


if __name__ == "__main__":
    app.run()

Overwriting orchestrator_agent.py


#### Launch the agent:

Again, we use the configure_runtime helper function to create the .bedrock_agentcore.yaml, .dockerignore, and Dockerfile required for the agent deployment. Then we call .launch() on the Orchestrator Agent runtime which pushes the image to ECR and creates the AgentCore Runtime in the AWS environment. 

In [23]:
_, orchestrator_agentcore_runtime = configure_runtime("orchestrator_agent_" + user_name, orchestrator_iam_role, "orchestrator_agent.py")
orchestrator_launch_result = orchestrator_agentcore_runtime.launch()

Entrypoint parsed: file=/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/orchestrator_agent/orchestrator_agent.py, bedrock_agentcore_name=orchestrator_agent
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: orchestrator_agent_eric_fu
Memory disabled
Network mode: PUBLIC


📄 Using existing Dockerfile: 
/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-ag
ents/01-multi-runtimes-with-boto3/orchestrator_agent/Dockerfile

Generated .dockerignore: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/orchestrator_agent/.dockerignore
Keeping 'orchestrator_agent_eric_fu' as default agent
Bedrock AgentCore configured: /Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/orchestrator_agent/.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'orchestrator_agent_eric_fu' to account 372080370602 (us-west-2)
Generated image tag: 20260406-000632-492
Setting

✅ Reusing existing ECR repository: 372080370602.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-orchestrator_agent_eric_fu


Getting or creating CodeBuild execution role for agent: orchestrator_agent_eric_fu
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-5141f2e8d3
Reusing existing CodeBuild execution role: arn:aws:iam::372080370602:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-5141f2e8d3
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: orchestrator_agent_eric_fu/source.zip
Updated CodeBuild project: bedrock-agentcore-orchestrator_agent_eric_fu-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.1s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 8.5s
🔄 DOWNLOAD_SOURCE started (total: 10s)
✅ DOWNLOAD_SOURCE completed in 2.1s
🔄 BUILD started (total: 12s)
✅ BUILD completed in 18.0s
🔄 POST_BUILD started (total: 30s)
✅ POST_BUILD completed in 11.7s
🔄 FINALIZING started (total: 41s)
✅ FINALIZING completed in 1.1s
🔄 COMPLETED started (total: 43s)
✅ CO

#### Test the agent

Now let's check the status of the Orchestrator Agent AgentCore Runtime and confirm it is ready for use.\

This time, we can use the `invoke_agent_with_boto3` function from our utils to test the Orchestrator Agent. Let's ask the orchestrator a question that should trigger the invocation of both the Tech Support and HR Agents.

In [24]:
import os
print(os.getcwd())

/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3/orchestrator_agent


In [36]:
status = check_status(orchestrator_agentcore_runtime)
print(status)

from invoke_agent_utils import invoke_agent_with_boto3


result = invoke_agent_with_boto3 (orchestrator_launch_result.agent_arn, "tell me about my benefits, also tell me how to connect a bluetooth mouse to my mac", user_name=user_name)

Retrieved Bedrock AgentCore status for: orchestrator_agent_eric_fu


READY
Invoking agent...
Processing streaming response...

I'll help you with both of those questions! Let me contact the HR agent about your benefits and the tech agent about connecting a Bluetooth mouse to your Mac.

[Executing: call_HR_agent]



[Executing: call_tech_agent]

Let me try that again with the user name parameter:

[Executing: call_HR_agent]



[Executing: call_tech_agent]

I apologize for the technical difficulties in retrieving that information. However, I can provide you with general guidance on both topics:

## Benefits Information
To learn about your specific employee benefits, I recommend:
- Contacting your HR department directly
- Checking your company's employee benefits portal or intranet
- Reviewing the benefits documentation you received during onboarding

Common employee benefits typically include health insurance, dental/vision coverage, retirement plans, paid time off, and wellness programs.

## Connecting a Bluetooth Mouse to Your Mac

Here are the steps:



## Cleanup (Optional)

Let's now clean up the AgentCore Runtime created

In [25]:
# set the current working directory to be the hr_agent folder
import os

os.chdir('..')
print(os.getcwd())

/Users/eric.fu/projects/xealth/agentcore-samples/01-tutorials/01-AgentCore-runtime/03-advanced-concepts/05-multi-agents/01-multi-runtimes-with-boto3


In [26]:

print(orchestrator_launch_result.ecr_uri, orchestrator_launch_result.agent_id, orchestrator_launch_result.ecr_uri.split('/')[1])
print(hr_launch_result.ecr_uri, hr_launch_result.agent_id, hr_launch_result.ecr_uri.split('/')[1])
print(tech_launch_result.ecr_uri, tech_launch_result.agent_id, tech_launch_result.ecr_uri.split('/')[1])

372080370602.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-orchestrator_agent_eric_fu:20260406-000632-492 orchestrator_agent_eric_fu-90pQmg5Uve bedrock-agentcore-orchestrator_agent_eric_fu:20260406-000632-492
372080370602.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-hr_agent_eric_fu:20260405-235858-827 hr_agent_eric_fu-Vwau8T3bik bedrock-agentcore-hr_agent_eric_fu:20260405-235858-827
372080370602.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-tech_agent_eric_fu:20260405-235144-806 tech_agent_eric_fu-SQgUotG3ed bedrock-agentcore-tech_agent_eric_fu:20260405-235144-806


In [36]:
def clean_up_agent_runtimes(launch_result):
    print(f"Cleaning up agent runtimes and ECR repositories...{launch_result.ecr_uri}")
    agentcore_control_client = boto3.client(
        'bedrock-agentcore-control',
        region_name=os.getenv("AWS_DEFAULT_REGION")
    )
    ecr_client = boto3.client(
        'ecr',
        region_name=os.getenv("AWS_DEFAULT_REGION")

    )
    runtime_delete_response = agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )

    response = ecr_client.delete_repository(
        repositoryName=launch_result.ecr_uri.split('/')[1].split(':')[0],
        force=True
    )

    return response

def delete_iam_roles(agentcore_iam_role):
    iam_client = boto3.client('iam')
    policies = iam_client.list_role_policies(
        RoleName=agentcore_iam_role['Role']['RoleName'],
        MaxItems=100
    )

    for policy_name in policies['PolicyNames']:
        iam_client.delete_role_policy(
            RoleName=agentcore_iam_role['Role']['RoleName'],
            PolicyName=policy_name
        )
    iam_response = iam_client.delete_role(
        RoleName=agentcore_iam_role['Role']['RoleName']
    )
    return iam_response

In [38]:
hr_launch_result.agent_id

'hr_agent_eric_fu-Vwau8T3bik'

In [ ]:
print(clean_up_agent_runtimes(hr_launch_result))
print(clean_up_agent_runtimes(tech_launch_result))
print(clean_up_agent_runtimes(orchestrator_launch_result))
print(delete_iam_roles(tech_agent_iam_role))
print(delete_iam_roles(hr_agent_iam_role))
print(delete_iam_roles(orchestrator_iam_role))

Cleaning up agent runtimes and ECR repositories...372080370602.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-tech_agent_eric_fu:20260405-235144-806
{'repository': {'repositoryArn': 'arn:aws:ecr:us-west-2:372080370602:repository/bedrock-agentcore-tech_agent_eric_fu', 'registryId': '372080370602', 'repositoryName': 'bedrock-agentcore-tech_agent_eric_fu', 'repositoryUri': '372080370602.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-tech_agent_eric_fu', 'createdAt': datetime.datetime(2026, 4, 5, 15, 54, 49, 646000, tzinfo=tzlocal()), 'imageTagMutability': 'MUTABLE'}, 'ResponseMetadata': {'RequestId': 'de2d466c-b1ce-4d18-934c-221034036bc8', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'de2d466c-b1ce-4d18-934c-221034036bc8', 'date': 'Mon, 06 Apr 2026 00:11:51 GMT', 'content-type': 'application/x-amz-json-1.1', 'content-length': '361', 'connection': 'keep-alive'}, 'RetryAttempts': 0}}
Cleaning up agent runtimes and ECR repositories...372080370602.dkr.ecr.us-west-2.amazona

# Congratulations!